# Final table v4: vote of 4 models

Self-contained: it only reads the data folder and the template, trains every model on all calibration
batches and writes the leaderboard table. Nothing is sent to the leaderboard.

**Process A** — a test batch is **rejected** when at least **2 of these 4** models flag it:

| # | Model |
|---|---|
| 1 | RDST (1000 points) + spread block, spread weight 2, ridge alpha 10 | 
| 2 | MiniRocket (500 points) + spread block, spread weight 2, ridge alpha 0.001 |
| 3 | MiniRocket (500 points) + summary block, summary weight 2, ridge alpha 0.1 |
| 4 | Windows: 60-row windows, MiniRocket + ridge |

**Process B** — RDST (1000 points) + spread block, weight 1, ridge alpha 0.001, score > 0.

Every ridge is `RidgeClassifier(class_weight="balanced")`. Seed 0 everywhere. Run time: a few minutes
(the RDST transforms are the slow part).

In [2]:
%pip install numpy==2.3.5 pandas==3.0.6 scikit-learn==1.9.1 aeon==1.6.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from aeon.transformations.collection.convolution_based import MiniRocket
from aeon.transformations.collection.shapelet_based import RandomDilatedShapeletTransform
from sklearn.linear_model import RidgeClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED = 0

# first data folder that exists
CANDIDATES = [Path("../../Project 2 - BRPMM"), Path("../Project 2 - BRPMM"), Path("Project 2 - BRPMM"),
              Path("../input-data/Project 2 - BRPMM"), Path("input-data/Project 2 - BRPMM")]
DATA = next(p for p in CANDIDATES if (p / "BRPMM_template.csv").exists())
OUT = Path("../outputs") if Path("../outputs").exists() else Path("outputs")
OUT.mkdir(exist_ok=True)
print("data:", DATA.resolve())

data: /Users/kikocaldas/Desktop/folder/Project 2 - BRPMM


## Data

In [4]:
def load(process, part):
    """List of (rows, sensors) arrays, labels (1 = rejected, calibration only) and batch ids."""
    d = DATA / f"process {process}" / f"{part} data"
    df = pd.read_csv(d / f"{part}_data.csv", dtype={"batch_id": str})
    sensors = [c for c in df.columns if c != "batch_id"]
    groups = {b: g[sensors].to_numpy(float) for b, g in df.groupby("batch_id")}
    if part == "calibration":
        lab = pd.read_csv(d / "calibration_labels.csv", dtype={"batch_id": str})
        ids, y = list(lab.batch_id), lab.label.eq("rejected").astype(int).to_numpy()
    else:
        ids, y = sorted(groups), None
    return [groups[b] for b in ids], y, ids


cal_A, y_A, _ = load("A", "calibration")
test_A, _, test_ids_A = load("A", "test")
cal_B, y_B, _ = load("B", "calibration")
test_B, _, test_ids_B = load("B", "test")
print(f"A: {len(cal_A)} calibration ({y_A.sum()} rejected), {len(test_A)} test | "
      f"B: {len(cal_B)} calibration ({y_B.sum()} rejected), {len(test_B)} test")

A: 161 calibration (40 rejected), 161 test | B: 40 calibration (5 rejected), 40 test


## Features

In [5]:
def resample(a, length):
    """(rows, sensors) -> (sensors, length): slice means, or interpolation for short batches."""
    n = len(a)
    if n < length:
        old, new = np.linspace(0, 1, n), np.linspace(0, 1, length)
        return np.stack([np.interp(new, old, a[:, j]) for j in range(a.shape[1])])
    idx = np.arange(n) * length // n
    sums = np.zeros((length, a.shape[1]))
    np.add.at(sums, idx, a)
    return (sums / np.bincount(idx, minlength=length)[:, None]).T


def spread_block(arrays):
    """Per sensor: log of std, of mean |step| and of range."""
    rows = []
    for a in arrays:
        st, step, rng = a.std(0), np.abs(np.diff(a, axis=0)).mean(0), a.max(0) - a.min(0)
        rows.append(np.log(np.concatenate([st, step, rng]) + 1e-6))
    return np.asarray(rows)


def summary_block(arrays):
    """Per sensor: mean, std, min, max, median, first, last, std of the first difference, slope; log length."""
    rows = []
    for a in arrays:
        t = np.linspace(-1, 1, len(a))
        slope = (t[:, None] * (a - a.mean(0))).sum(0) / (t ** 2).sum()
        d = np.diff(a, axis=0)
        rows.append(np.concatenate([a.mean(0), a.std(0), a.min(0), a.max(0), np.median(a, 0), a[0], a[-1],
                                    d.std(0), slope, [np.log(len(a))]]))
    return np.asarray(rows)


def series_features(kind, length, cal, test, y):
    """RDST or MiniRocket features: series resampled to `length`, each sensor standardised with
    calibration statistics, transform fitted on calibration only."""
    Xtr = np.stack([resample(a, length) for a in cal])
    Xte = np.stack([resample(a, length) for a in test])
    mu, sd = Xtr.mean(axis=(0, 2), keepdims=True), Xtr.std(axis=(0, 2), keepdims=True)
    sd[sd == 0] = 1.0
    Xtr, Xte = (Xtr - mu) / sd, (Xte - mu) / sd
    if kind == "rdst":
        t = RandomDilatedShapeletTransform(random_state=SEED, n_jobs=1)
        Ftr = t.fit_transform(Xtr, y)
    else:
        t = MiniRocket(random_state=SEED)
        Ftr = t.fit_transform(Xtr)
    Fte = t.transform(Xte)
    clean = lambda F: np.nan_to_num(np.asarray(F, dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    return clean(Ftr), clean(Fte)

## Model helpers

In [6]:
def two_block_scores(block1, block2, weight, alpha, y):
    """Standardise each block, scale it by weight / sqrt(number of features) (block 1 weight 1),
    one balanced ridge; returns the test scores."""
    parts_tr, parts_te = [], []
    for (Ftr, Fte), w in ((block1, 1.0), (block2, weight)):
        sc = StandardScaler().fit(Ftr)
        k = w / np.sqrt(Ftr.shape[1])
        parts_tr.append(sc.transform(Ftr) * k)
        parts_te.append(sc.transform(Fte) * k)
    model = RidgeClassifier(alpha=alpha, class_weight="balanced").fit(np.hstack(parts_tr), y)
    return model.decision_function(np.hstack(parts_te))


def top(scores, n):
    """Flag the n test batches with the highest scores."""
    return scores >= np.sort(scores)[::-1][n - 1]


def windows(a, size=60):
    """Cut a batch into consecutive windows of `size` rows -> (windows, sensors, size)."""
    if len(a) < size:
        return resample(a, size)[None]
    return np.stack([a[s:s + size].T for s in range(0, len(a) - size + 1, size)])


def windows_model(cal, y, test, alpha=1000.0):
    """Windows of 60 rows, MiniRocket + balanced ridge on the windows; a batch is rejected when more
    than half of its windows score > 0."""
    def stack(arrays):
        parts = [windows(a) for a in arrays]
        return np.concatenate(parts), np.repeat(np.arange(len(arrays)), [len(p) for p in parts])
    X, g = stack(cal)
    mu, sd = X.mean(axis=(0, 2), keepdims=True), X.std(axis=(0, 2), keepdims=True) + 1e-12
    rocket = MiniRocket(random_state=0, n_jobs=-1).fit((X - mu) / sd)
    sc = StandardScaler()
    model = RidgeClassifier(alpha=alpha, class_weight="balanced").fit(sc.fit_transform(rocket.transform((X - mu) / sd)), y[g])
    Xs, gs = stack(test)
    s = model.decision_function(sc.transform(rocket.transform((Xs - mu) / sd)))
    return np.bincount(gs, weights=(s > 0).astype(float)) / np.bincount(gs) > 0.5

## Process A: train the 4 models

In [7]:
rdst1000_A = series_features("rdst", 1000, cal_A, test_A, y_A)
mini500_A = series_features("minirocket", 500, cal_A, test_A, y_A)
spread_A = (spread_block(cal_A), spread_block(test_A))
summary_A = (summary_block(cal_A), summary_block(test_A))

vote1 = top(two_block_scores(rdst1000_A, spread_A, weight=2.0, alpha=10, y=y_A), 57)
vote2 = top(two_block_scores(mini500_A, spread_A, weight=2.0, alpha=0.001, y=y_A), 13)
vote3 = top(two_block_scores(mini500_A, summary_A, weight=2.0, alpha=0.1, y=y_A), 24)
vote4 = windows_model(cal_A, y_A, test_A, alpha=1000.0)
for name, v in [("RDST + spread", vote1), ("MiniRocket + spread", vote2), ("MiniRocket + summary", vote3), ("windows", vote4)]:
    print(f"{name:22s} flags {v.sum()} test batches")

RDST + spread          flags 57 test batches
MiniRocket + spread    flags 13 test batches
MiniRocket + summary   flags 24 test batches
windows                flags 29 test batches


## Process A: vote (at least 2 of 4)

In [8]:
votes = vote1.astype(int) + vote2 + vote3 + vote4
rejected_A = votes >= 2
print(f"process A: {rejected_A.sum()} of {len(test_ids_A)} rejected")

process A: 34 of 161 rejected


## Process B: RDST + spread

In [9]:
rdst1000_B = series_features("rdst", 1000, cal_B, test_B, y_B)
spread_B = (spread_block(cal_B), spread_block(test_B))
rejected_B = two_block_scores(rdst1000_B, spread_B, weight=1.0, alpha=0.001, y=y_B) > 0
print(f"process B: {rejected_B.sum()} of {len(test_ids_B)} rejected")

process B: 5 of 40 rejected


## Final table (template order)

In [10]:
template = pd.read_csv(DATA / "BRPMM_template.csv", dtype={"batch_id": str})
template.columns = template.columns.str.strip()
template["batch_id"] = template.batch_id.str.strip()
labels = pd.concat([
    pd.DataFrame({"batch_id": test_ids_A, "label": np.where(rejected_A, "rejected", "accepted")}),
    pd.DataFrame({"batch_id": test_ids_B, "label": np.where(rejected_B, "rejected", "accepted")}),
])
table = template[["batch_id", "project"]].merge(labels, on="batch_id", how="left")
assert len(table) == len(template) and table.label.notna().all()
table.to_csv(OUT / "final_v4.csv", index=False)
print(table.groupby("project").label.value_counts())  # expected: A 34 rejected, B 5 rejected
print("written to", (OUT / "final_v4.csv").resolve(), "- not submitted")

project  label   
A        accepted    127
         rejected     34
B        accepted     35
         rejected      5
Name: count, dtype: int64
written to /Users/kikocaldas/Desktop/folder/SDSC-STEM4Health-group2/outputs/final_v4.csv - not submitted
